In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain.tools import tool
import requests

In [ ]:
from langchain.agents import create_agent

In [6]:
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["WEATHERSTACK_API_KEY"] = os.getenv("WEATHERSTACK_API_KEY")

In [7]:
search_tool = TavilySearch(max_results=3)


In [8]:
@tool
def get_weather(city: str) -> str:
    """Fetch current weather information for a city"""

    api_key = os.getenv("WEATHERSTACK_API_KEY")

    if not api_key:
        return "WEATHERSTACK_API_KEY is not set"

    url = (
        f"http://api.weatherstack.com/current"
        f"?access_key={api_key}"
        f"&query={city}"
    )

    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}: {data}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [9]:
result = search_tool.invoke("what is the latest news on AI?")
result

{'query': 'what is the latest news on AI?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.artificialintelligence-news.com',
   'title': 'AI News | Latest News | Insights Powering AI-Driven Business Growth',
   'content': 'October 30, 2025\n\n### Malaysia launches Ryt Bank, its first AI-powered bank\n\nFinance AI\n\nAugust 26, 2025\n\n### Google’s Veo 3 AI video creation tools are now widely available\n\nAI in Action\n\nJuly 29, 2025\n\n#### Computer Vision\n\n### Microsoft’s Majorana 2 quantum chip is also a case study for agentic AI in R&D\n\nInside AI\n\nJune 3, 2026\n\n### US and Japan announce sweeping AI and tech collaboration\n\nArtificial Intelligence\n\nApril 11, 2024\n\n### UK and Canada sign AI compute agreement [...] August 18, 2026\n\n### Okta targets AI agent token costs with MCP scoping\n\nData Engineering & MLOps\n\nAugust 13, 2026\n\n### Red Hat, NVIDIA, IBM back project turning AI policy into code\n\nGovernance, Regulat

In [10]:
llm = ChatGroq(
    model = "openai/gpt-oss-20b",
    api_key = os.environ["GROQ_API_KEY"]
)

In [11]:
response = llm.invoke("what year is it?")
response

AIMessage(content='It’s the year **2026**.', additional_kwargs={'reasoning_content': "We need to respond. The current date is given: 2026-09-07. So it's year 2026. Probably answer: 2026."}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 76, 'total_tokens': 128, 'completion_time': 0.053907828, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.003771488, 'prompt_tokens_details': None, 'queue_time': 0.341057401, 'total_time': 0.057679316}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_7d448090ba', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07b23-30bd-7ce0-a268-47c73de6c067-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 52, 'total_tokens': 128, 'output_token_details': {'reasoning': 34}})

In [12]:
tools = [search_tool,get_weather]

In [13]:
agent = create_agent(
    model = llm,
    tools = tools
)

In [ ]:
response = agent.invoke({
    "messages":[
        {"role":"user","content":"Find the current weather in ongole?"}
    ]
})

In [15]:
print(response["messages"][-1].content)

**Current Weather in Ongole**

- **Temperature:** 38 °C  
- **Weather:** Overcast  
- **Humidity:** 33%


In [16]:
for msg in response["messages"]:
    print(msg.content)

Find the current weather in ongole?

City: Ongole
Temperature: 38°C
Weather: Overcast 
Humidity: 33%
**Current Weather in Ongole**

- **Temperature:** 38 °C  
- **Weather:** Overcast  
- **Humidity:** 33%
